# Linear regression: from a fitted line to trustworthy prediction

**Lecture 9 · Notebook 01 · CMOR 438 / INDE 577**  
**Core:** 150 minutes · **Practice:** 50 minutes · **Extension:** 35+ minutes

Linear regression is both a mathematical object we can derive and a complete supervised-learning system we can audit. We will first fit a line by hand, visualize what least squares minimizes, and recover the solution from the normal equations. We will then place the same estimator inside a leakage-safe scientific prediction pipeline.

The recurring question is:

> **What was minimized, on which observations, and what evidence supports prediction on a new observation?**

## How to use this notebook

Work in two modes:

1. **Mathematical mode:** keep track of symbols, dimensions, residuals, and the objective. Derive each displayed result before reading the code.
2. **Systems mode:** keep track of which rows and fitted transformations are allowed to influence each decision.

Run every cell from top to bottom in the **Rice DSM** kernel. The board calculation and the full pipeline use different views of the same real clinical dataset so the mathematics never depends on invented observations.

**Prerequisites:** functions, sums, vectors, matrices, NumPy, pandas, and the machine-learning landscape notebook.

## Learning objectives

By the end, you should be able to:

- formulate regression as learning a function from labeled examples;
- distinguish an observation, fitted value, prediction, error, and residual;
- derive ordinary least squares for one feature and express it in matrix form;
- explain geometrically what squared-error minimization does;
- explain why training, validation, and test data have different roles;
- build preprocessing and linear regression as one fitted pipeline;
- state precisely when scaling changes predictions and when it changes optimization;
- interpret MAE, RMSE, and $R^2$ without calling any of them “accuracy”; and
- diagnose leakage, misspecification, subgroup error, and unsupported causal claims.

## Why this matters in industry

A library can return coefficients in one line, but a professional must still answer:

- What quantity is available at **prediction time**?
- What population will receive predictions?
- Which loss defines “best,” and whose errors does it emphasize?
- Which information was allowed to influence preprocessing and model choice?
- How large are errors on genuinely unseen cases, in physical units?
- Where does the model fail, and what action follows a prediction?

Linear regression is an unusually good first model because every layer—geometry, probability, numerical linear algebra, evaluation, and deployment—remains visible.

## Historical context: least squares was a scientific computing method first

Linear regression connects modern prediction systems to more than two centuries of science:

- **1805:** Adrien-Marie Legendre published the method of least squares while studying comet orbits.
- **1809:** Carl Friedrich Gauss published a probabilistic treatment and reported having used the principle earlier.
- **Late nineteenth century:** regression terminology developed through studies of heredity; that historical interpretation is not the definition of today's model.
- **Twentieth century onward:** linear models became central to statistical inference, signal processing, calibration, control, econometrics, and predictive modeling.

The original problem remains recognizable: several noisy observations constrain fewer unknown parameters, and no parameter choice satisfies every equation. Least squares turns the inconsistency into an optimization problem. Today we must additionally ask whether the measurements, sampling process, and future use justify the fitted relationship.

In [ ]:
from __future__ import annotations

import sqlite3
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rice_dsm.paths import course_database_path
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 438577
rng = np.random.default_rng(RANDOM_SEED)

## 1. The supervised regression problem

A labeled dataset contains pairs

$$
\mathcal{D}=\{(\boldsymbol{x}_i,y_i)\}_{i=1}^{n},
$$

where $\boldsymbol{x}_i\in\mathbb{R}^p$ is a feature vector and $y_i\in\mathbb{R}$ is a quantitative target. Learning produces a function $\widehat f$ so that, for a new feature vector $\boldsymbol{x}_*$,

$$\widehat y_* = \widehat f(\boldsymbol{x}_*)$$

is useful for an explicitly stated population and action.

The hat matters: $\widehat f$ and $\widehat y$ were estimated from finite data. The target $y_*$ generally remains uncertain even if the conditional mean is perfectly modeled.

### A tiny board example

We query baseline measurements from 442 patients and a quantitative disease-progression response measured one year later from the repository's SQLite teaching database. Its catalog records that some feature meanings are not completely explicit. That limitation is part of the lesson, not a footnote to hide.

The versioned database works offline. We connect in read-only mode, inspect its metadata, and issue SQL instead of importing a ready-made dataframe. We split immediately—before choosing examples or drawing target-guided plots—then use twelve training observations for a transparent board calculation.

The kernel's current working directory is runtime state; it need not equal the notebook directory or repository root. Therefore production-quality code should not assume that `data/...` is relative to one particular launch location. `course_database_path()` locates the repository and returns an absolute path, while `Path.as_uri()` expresses that path correctly on Windows, macOS, and Linux.

In [ ]:
database_path = course_database_path()
database_uri = f"{database_path.as_uri()}?mode=ro"
course_database = sqlite3.connect(database_uri, uri=True)
diabetes_metadata = pd.read_sql_query(
    "SELECT * FROM dataset_catalog WHERE dataset_id = ?",
    course_database,
    params=("diabetes",),
)
diabetes = pd.read_sql_query(
    """
    SELECT observation_id, age, sex, bmi, bp, s1, s2, s3, s4, s5, s6,
           disease_progression
    FROM diabetes_observations
    ORDER BY observation_id
    """,
    course_database,
    index_col="observation_id",
)
development_rows, test_rows = train_test_split(
    diabetes.index, test_size=0.20, random_state=RANDOM_SEED
)
train_rows, validation_rows = train_test_split(
    development_rows, test_size=0.25, random_state=RANDOM_SEED
)
train_data = diabetes.loc[train_rows].copy()
validation_data = diabetes.loc[validation_rows].copy()
test_data = diabetes.loc[test_rows].copy()

board_data = train_data.iloc[:12]
x_board = board_data["bmi"].to_numpy()
y_board = board_data["disease_progression"].to_numpy()
centered_x = x_board - x_board.mean()
centered_y = y_board - y_board.mean()
board_slope = centered_x @ centered_y / (centered_x @ centered_x)
board_intercept = y_board.mean() - board_slope * x_board.mean()

assert x_board.shape == y_board.shape == (12,)
assert np.unique(x_board).size > 1
assert (len(train_data), len(validation_data), len(test_data)) == (264, 89, 89)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x_board, y_board, s=65)
ax.set(
    xlabel="baseline body mass index, $x$ (kg/m²)",
    ylabel="one-year disease-progression score, $y$",
    title="Twelve real training observations",
)
plt.show()

## 2. The affine model and its residuals

With one feature, linear regression uses the affine function

$$
\widehat y_i = b + w x_i.
$$

The parameter $b$ is the **intercept** and $w$ is the **slope**. “Linear regression” conventionally includes the intercept even though the map is mathematically affine.

For a proposed line, the residual on an observed training point is

$$
e_i = y_i-\widehat y_i.
$$

A residual is observable after fitting because $y_i$ is known. A future prediction error $y_*-\widehat y_*$ is unknown until the future outcome arrives. The next figure draws residuals as vertical signed distances.

In [ ]:
candidate_lines = [
    (board_intercept + 60, 0.60 * board_slope, "shifted candidate"),
    (board_intercept, board_slope, "least-squares line"),
    (board_intercept - 80, 1.40 * board_slope, "too steep"),
]
x_domain = np.linspace(x_board.min() - 1, x_board.max() + 1, 120)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True, sharey=True)
for ax, (intercept, slope, label) in zip(axes, candidate_lines, strict=True):
    fitted = intercept + slope * x_board
    squared_error = np.sum((y_board - fitted) ** 2)
    ax.scatter(x_board, y_board, color="black", zorder=3)
    ax.plot(x_domain, intercept + slope * x_domain, color="tab:blue")
    for x_value, observed, predicted in zip(x_board, y_board, fitted, strict=True):
        ax.vlines(x_value, predicted, observed, color="crimson", alpha=0.7)
    ax.set_title(f"{label}\nSSE = {squared_error:.3f}")
    ax.set_xlabel("body mass index (kg/m²)")
axes[0].set_ylabel("disease-progression score")
plt.tight_layout()
plt.show()

## 3. Ordinary least squares chooses among all lines

Ordinary least squares (OLS) minimizes the **sum of squared errors**

$$
Q(b,w)=\sum_{i=1}^{n}\left[y_i-(b+w x_i)\right]^2.
$$

The mean squared error is $Q/n$ and has the same minimizer. Squaring makes positive and negative residuals contribute without canceling, penalizes large residuals strongly, and produces a differentiable convex objective. Those are modeling choices—not universal laws.

The horizontal axes below are parameter space, not data space. Every point $(b,w)$ represents one entire line in the previous figure; height/color represents that line's loss.

In [ ]:
intercept_grid = np.linspace(board_intercept - 180, board_intercept + 180, 180)
slope_grid = np.linspace(board_slope - 12, board_slope + 12, 180)
B, W = np.meshgrid(intercept_grid, slope_grid)
objective_grid = np.sum(
    (y_board[:, None, None] - (B[None, :, :] + W[None, :, :] * x_board[:, None, None])) ** 2,
    axis=0,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
surface = axes[0].contourf(B, W, objective_grid, levels=30, cmap="viridis")
axes[0].scatter(board_intercept, board_slope, color="crimson", marker="*", s=140, label="OLS minimum")
axes[0].set(xlabel="intercept $b$", ylabel="slope $w$", title="$Q(b,w)$ in parameter space")
axes[0].legend()
fig.colorbar(surface, ax=axes[0], label="sum of squared errors")

axes[1].scatter(x_board, y_board, color="black", label="observations")
axes[1].plot(x_domain, board_intercept + board_slope * x_domain, color="crimson", label="OLS line")
axes[1].set(xlabel="body mass index (kg/m²)", ylabel="disease-progression score", title="The minimizing line in data space")
axes[1].legend()
plt.tight_layout()
plt.show()

### Deriving the one-feature solution

At the minimum of this convex quadratic, both partial derivatives vanish:

$$
\frac{\partial Q}{\partial b}=-2\sum_i(y_i-b-wx_i)=0,
\qquad
\frac{\partial Q}{\partial w}=-2\sum_i x_i(y_i-b-wx_i)=0.
$$

The first equation gives $b=\bar y-w\bar x$. Substituting into the second gives

$$
\widehat w=
\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}
     {\sum_i(x_i-\bar x)^2},
\qquad
\widehat b=\bar y-\widehat w\bar x.
$$

The denominator explains an identifiability condition: if every $x_i$ is equal, no observed change in $x$ exists from which to estimate a slope.

In [ ]:
centered_x = x_board - x_board.mean()
centered_y = y_board - y_board.mean()
slope_by_formula = centered_x @ centered_y / (centered_x @ centered_x)
intercept_by_formula = y_board.mean() - slope_by_formula * x_board.mean()

design_board = np.column_stack([np.ones_like(x_board), x_board])
coefficient_by_lstsq, residual_sums, rank, singular_values = np.linalg.lstsq(
    design_board, y_board, rcond=None
)
sklearn_board = LinearRegression().fit(x_board.reshape(-1, 1), y_board)

assert rank == 2
assert np.allclose(coefficient_by_lstsq, [intercept_by_formula, slope_by_formula])
assert np.isclose(sklearn_board.intercept_, intercept_by_formula)
assert np.isclose(sklearn_board.coef_[0], slope_by_formula)
print(f"fitted equation: y_hat = {intercept_by_formula:.4f} + {slope_by_formula:.4f} x")

### Matrix form and the normal equations

For $p$ features, place a column of ones and the feature columns in a design matrix:

$$
X=\begin{bmatrix}
1 & x_{11}&\cdots&x_{1p}\\
\vdots&\vdots&&\vdots\\
1 & x_{n1}&\cdots&x_{np}
\end{bmatrix},
\qquad
\boldsymbol{\beta}=\begin{bmatrix}b&w_1&\cdots&w_p\end{bmatrix}^{\!T}.
$$

Then $\widehat{\boldsymbol y}=X\boldsymbol\beta$ and

$$
Q(\boldsymbol\beta)=\|\boldsymbol y-X\boldsymbol\beta\|_2^2,
\qquad
\nabla Q=2X^T(X\boldsymbol\beta-\boldsymbol y).
$$

Setting the gradient to zero yields the normal equations

$$X^TX\widehat{\boldsymbol\beta}=X^T\boldsymbol y.$$

The mnemonic $(X^TX)^{-1}X^T\boldsymbol y$ requires full column rank, and explicitly forming an inverse is poor numerical practice. Production libraries use stable least-squares solvers such as QR or SVD-based methods. `np.linalg.lstsq` also exposes rank and singular values, which help diagnose non-identifiability.

### Projection is the geometric theorem behind OLS

The vector $X\widehat{\boldsymbol\beta}$ lies in the column space $\mathcal{C}(X)$. At a least-squares solution,

$$
X^T\bigl(\boldsymbol y-X\widehat{\boldsymbol\beta}\bigr)=\boldsymbol 0,
$$

so the residual vector is orthogonal to every column of $X$. For any alternative coefficient vector $\boldsymbol v$,

$$
\|\boldsymbol y-X\boldsymbol v\|_2^2
=
\|\boldsymbol y-X\widehat{\boldsymbol\beta}\|_2^2
+
\|X\widehat{\boldsymbol\beta}-X\boldsymbol v\|_2^2.
$$

This Pythagorean identity proves that the projection minimizes squared distance. It also explains why fitted values can be unique even when coefficients are not: different coefficient vectors may map to the same point in $\mathcal{C}(X)$.

**Board checkpoint:** show that the intercept column implies $\sum_i e_i=0$, and that the feature column implies $\sum_i x_i e_i=0$ in one-feature regression.

### What does the fitted line estimate?

A common statistical model is

$$Y=b+wX+\varepsilon, \qquad \mathbb{E}[\varepsilon\mid X]=0.$$

Under that conditional-mean assumption, the affine function represents $\mathbb{E}[Y\mid X=x]$. A new observation still contains irreducible variation $\varepsilon$. Linearity can be useful as an approximation even when it is not literally true.

Normal errors are **not** required to calculate OLS predictions. They enter particular finite-sample inference results. Independence, constant variance, representative sampling, correct functional form, and measurement quality are separate assumptions and should not be collapsed into one vague phrase.

### Why squared loss targets the conditional mean

At a fixed feature value $X=x$, suppose we may predict any number $a$. Its conditional squared-error risk is

$$
q(a)=\mathbb{E}\left[(Y-a)^2\mid X=x\right].
$$

Differentiating with respect to $a$ gives

$$
q'(a)=2\left(a-\mathbb{E}[Y\mid X=x]\right),
$$

so the unique minimizer is the conditional mean when the required moment exists. Absolute loss instead targets a conditional median. This is why “regression” is incomplete without its loss: the loss helps define the population quantity being estimated.

The fitted affine model approximates that conditional mean inside a restricted function class. A new patient's outcome still varies around the conditional mean, and the training sample makes the fitted approximation uncertain.

## 4. From fitting to generalization

Minimizing training loss answers: “Which line best fits these observed labels under squared loss?” Machine learning usually asks the harder question: “How well will the fitted procedure predict labels from the intended future population?”

We therefore separate data by role:

- **Training data** estimate preprocessing and model parameters.
- **Validation data** compare choices and diagnose errors.
- **Test data** provide one final estimate after choices are frozen.

Training error is feedback used by the algorithm. It is therefore optimistically biased as evidence about unseen cases. Repeatedly consulting a validation set also adapts decisions to it; an untouched test set protects a final boundary.

## Worked example: predicting one-year disease progression

One row is one patient. Ten baseline variables—age, sex, body mass index, average blood pressure, and six serum measurements—are potential features; the target is a quantitative measure of disease progression one year later.

This is a useful **methods dataset**, not a deployment-ready clinical product. It is small; the precise target scale and some feature semantics are incompletely documented; and the bundled data do not identify collection site, time, repeated patients, missingness process, or representativeness. We can study regression mechanics without pretending the evidence supports clinical use.

In [ ]:
display(diabetes_metadata.T)
assert diabetes.shape == (442, 11)
assert diabetes.columns[-1] == "disease_progression"
assert not diabetes.isna().any().any()
diabetes.head()

### The evidence boundary was created before exploration

We reserved rows in the first data cell, before plotting relationships or selecting the twelve-point example. The random split is useful for teaching but assumes observations are exchangeable. A real clinical evaluation would usually need site-, patient-, and time-aware separation; this dataset does not provide the identifiers required to audit those boundaries.

In [ ]:
assert set(train_data.index).isdisjoint(validation_data.index)
assert set(train_data.index).isdisjoint(test_data.index)
assert set(validation_data.index).isdisjoint(test_data.index)
print({"training": len(train_data), "validation": len(validation_data), "test": len(test_data)})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].scatter(train_data["bmi"], train_data["disease_progression"], alpha=0.55)
axes[0, 0].set(xlabel="BMI (kg/m²)", ylabel="progression score", title="BMI and outcome")
axes[0, 1].scatter(train_data["bp"], train_data["disease_progression"], alpha=0.55, color="tab:orange")
axes[0, 1].set(xlabel="average blood pressure", ylabel="progression score", title="Blood pressure and outcome")
axes[1, 0].hist(train_data["disease_progression"], bins=22, edgecolor="white")
axes[1, 0].set(xlabel="progression score", ylabel="count", title="Training-target distribution")
correlation = train_data.corr(numeric_only=True)
image = axes[1, 1].imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
axes[1, 1].set_xticks(range(len(correlation)), correlation.columns, rotation=90)
axes[1, 1].set_yticks(range(len(correlation)), correlation.index)
axes[1, 1].set_title("Training correlations")
fig.colorbar(image, ax=axes[1, 1], shrink=0.75)
plt.tight_layout()
plt.show()

## 5. Begin with a baseline

A mean predictor assigns every input the mean training target. It is an experimental control: unless a learned model improves on it under the same evaluation, the features have not demonstrated predictive value.

In [ ]:
target_name = "disease_progression"
numeric_features = [
    column for column in diabetes.columns if column != "disease_progression"
]
feature_names = numeric_features

baseline = DummyRegressor(strategy="mean")
baseline.fit(train_data[feature_names], train_data[target_name])
baseline_validation_prediction = baseline.predict(validation_data[feature_names])
baseline_validation_mae = mean_absolute_error(validation_data[target_name], baseline_validation_prediction)

assert np.allclose(baseline_validation_prediction, train_data[target_name].mean())
print(f"validation baseline MAE: {baseline_validation_mae:.2f} score points")

## 6. Multiple features and one fitted pipeline

The model is now

$$
\widehat y=b+\sum_{j=1}^{10}w_jx_j.
$$

The ten numeric columns are standardized using means and standard deviations learned from **training data only**. The transformer and estimator form one `Pipeline` so fitting, validation, and later prediction apply the same transformation contract. Mixed numeric/categorical data use the same pattern with additional `ColumnTransformer` branches; no artificial category is introduced here merely to demonstrate an API.

`LinearRegression` is still OLS after transformation. More columns do not change the mathematical objective; they change the representation and the space of candidate affine functions.

In [ ]:
preprocessing = ColumnTransformer(
    [
        ("numeric", StandardScaler(), numeric_features),
    ]
)
regression_pipeline = Pipeline(
    [("preprocessing", preprocessing), ("model", LinearRegression())]
)
regression_pipeline.fit(train_data[feature_names], train_data[target_name])

fitted_scaler = regression_pipeline.named_steps["preprocessing"].named_transformers_["numeric"]
assert np.allclose(fitted_scaler.mean_, train_data[numeric_features].mean().to_numpy())
assert regression_pipeline.predict(train_data[feature_names].iloc[:3]).shape == (3,)

### Scaling: say exactly what it does

Standardization replaces $x$ by $z=(x-\mu)/s$. The same one-feature prediction can be written

$$b+wx=(b+w\mu)+(ws)z.$$

Therefore, with an intercept and no regularization, invertible rescaling does not change the OLS prediction class; it changes coefficient coordinates. Scaling still matters because it:

- improves numerical conditioning when columns have very different magnitudes;
- makes gradient-based optimization behave more evenly;
- determines the meaning of regularization penalties; and
- prevents raw coefficient magnitudes from merely reflecting units.

The next experiment verifies prediction invariance and compares design-matrix condition numbers.

In [ ]:
raw_numeric_model = LinearRegression().fit(train_data[numeric_features], train_data[target_name])
scaled_numeric_model = make_pipeline(StandardScaler(), LinearRegression()).fit(
    train_data[numeric_features], train_data[target_name]
)
raw_prediction = raw_numeric_model.predict(validation_data[numeric_features])
scaled_prediction = scaled_numeric_model.predict(validation_data[numeric_features])

raw_design = np.column_stack([np.ones(len(train_data)), train_data[numeric_features].to_numpy()])
scaled_design = np.column_stack([
    np.ones(len(train_data)), StandardScaler().fit_transform(train_data[numeric_features])
])
raw_condition_number = np.linalg.cond(raw_design)
scaled_condition_number = np.linalg.cond(scaled_design)

assert np.allclose(raw_prediction, scaled_prediction, atol=1e-10)
assert scaled_condition_number < raw_condition_number
print(f"condition number: raw={raw_condition_number:.1f}, scaled={scaled_condition_number:.1f}")

## 7. Metrics are mathematical summaries of different costs

Given held-out errors $r_i=y_i-\widehat y_i$:

$$
\operatorname{MAE}=\frac1n\sum_i|r_i|,
\qquad
\operatorname{RMSE}=\sqrt{\frac1n\sum_i r_i^2},
$$

$$
R^2=1-\frac{\sum_i(y_i-\widehat y_i)^2}{\sum_i(y_i-\bar y)^2}.
$$

- **MAE** has target units and gives errors linear weight.
- **RMSE** has target units but gives large errors greater influence before taking the square root.
- **$R^2$** is dimensionless and compares squared error with a constant reference on the evaluated data. It can be negative on held-out data and is not “percent accurate.”

No metric defines acceptable performance without a domain threshold and an account of who bears each error.

In [ ]:
@dataclass(frozen=True)
class RegressionEvidence:
    """Complementary held-out regression metrics."""

    mae_score_points: float
    rmse_score_points: float
    r_squared: float


def evaluate_regression(observed: pd.Series | np.ndarray, predicted: np.ndarray) -> RegressionEvidence:
    """Compute MAE, RMSE, and R-squared.

    Parameters
    ----------
    observed
        Observed quantitative disease-progression scores.
    predicted
        Predicted capacities aligned with ``observed``.

    Returns
    -------
    RegressionEvidence
        Error metrics in score points and dimensionless R-squared.

    Raises
    ------
    ValueError
        If shapes differ or values are nonfinite.
    """

    observed_array = np.asarray(observed, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    if observed_array.shape != predicted_array.shape:
        raise ValueError("observed and predicted must have identical shapes")
    if not np.isfinite(observed_array).all() or not np.isfinite(predicted_array).all():
        raise ValueError("observed and predicted must contain finite values")
    return RegressionEvidence(
        mae_score_points=float(mean_absolute_error(observed_array, predicted_array)),
        rmse_score_points=float(np.sqrt(mean_squared_error(observed_array, predicted_array))),
        r_squared=float(r2_score(observed_array, predicted_array)),
    )


validation_prediction = regression_pipeline.predict(validation_data[feature_names])
validation_evidence = evaluate_regression(validation_data[target_name], validation_prediction)
assert validation_evidence.mae_score_points < baseline_validation_mae
assert validation_evidence.rmse_score_points >= validation_evidence.mae_score_points
assert validation_evidence.r_squared > 0.20
validation_evidence

## 8. Residuals test the story visually

A scalar score compresses structure. Plot held-out residuals against predictions and scientifically relevant groups. Curvature suggests missing nonlinear structure; a funnel suggests changing variance; a shifted subgroup suggests systematic error; a few extreme points may reveal measurement, population, or robustness problems.

A residual plot does not prove that assumptions hold, but visible structure can show that a simple story is inadequate.

In [ ]:
validation_audit = validation_data[["age", "bmi", target_name]].copy()
validation_audit["prediction"] = validation_prediction
validation_audit["residual"] = validation_audit[target_name] - validation_prediction
age_breaks = np.concatenate(
    ([-np.inf], train_data["age"].quantile([0.25, 0.50, 0.75]).to_numpy(), [np.inf])
)
validation_audit["age_band"] = pd.cut(
    validation_audit["age"], bins=age_breaks,
    labels=["youngest Q", "lower-middle Q", "upper-middle Q", "oldest Q"],
    include_lowest=True,
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.1))
axes[0].scatter(validation_data[target_name], validation_prediction, alpha=0.7)
diagonal = [validation_data[target_name].min(), validation_data[target_name].max()]
axes[0].plot(diagonal, diagonal, color="black", linestyle="--")
axes[0].set(xlabel="observed score", ylabel="predicted score", title="Observed versus predicted")
axes[1].scatter(validation_prediction, validation_audit["residual"], alpha=0.7)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(xlabel="predicted score", ylabel="observed − predicted", title="Residual structure")
validation_audit.boxplot(column="residual", by="age_band", ax=axes[2])
axes[2].tick_params(axis="x", rotation=25)
axes[2].set(xlabel="training-defined age quartile", ylabel="residual", title="Age-band audit")
plt.suptitle("")
plt.tight_layout()
plt.show()

validation_audit.groupby("age_band", observed=True)["residual"].agg(["count", "mean", "std"])

## Common failure modes

### Leakage

Fitting a scaler on all rows allows validation or test distribution information into training. Target-derived features are worse: a follow-up measurement cannot support a prediction claimed to occur at baseline.

### A split that breaks the deployment unit

Repeated measurements from one patient, instrument, site, or study on both sides can turn evaluation into recognition. Split the unit that must generalize. This dataset does not include every identifier needed to check those risks.

### Extrapolation

An affine equation produces a number outside the observed feature range, but that number is not automatically supported by evidence. Plot the training support and define an out-of-range policy.

### Collinearity and coefficient stories

Correlated columns can yield stable predictions but unstable individual coefficients. A coefficient is a conditional association under the fitted representation—not a causal effect of intervention.

### Optimizing the metric instead of the decision

A tiny average improvement may be irrelevant, while a rare large error may be unacceptable. Evaluation must reflect downstream costs and safety constraints.

## 9. Final test evaluation: open the envelope once

Features, preprocessing, estimator, and primary metrics are now frozen. Refit the chosen pipeline on all development data and evaluate the untouched test rows once.

In [ ]:
development_data = pd.concat([train_data, validation_data]).sort_index()
final_pipeline = Pipeline([("preprocessing", preprocessing), ("model", LinearRegression())])
final_pipeline.fit(development_data[feature_names], development_data[target_name])
test_prediction = final_pipeline.predict(test_data[feature_names])
test_evidence = evaluate_regression(test_data[target_name], test_prediction)

assert test_evidence.mae_score_points < baseline_validation_mae
assert test_evidence.r_squared > 0.40
course_database.close()
test_evidence

## Release evidence, not just coefficients

A releasable model artifact needs:

- feature names, order, units, valid ranges, category policy, and availability time;
- preprocessing and estimator serialized as one versioned object;
- training-data and code identity plus dependency versions;
- split definition, baseline, metrics, uncertainty, and subgroup diagnostics;
- intended population, prohibited uses, and extrapolation policy; and
- owner, monitoring signals, retraining trigger, and rollback path.

This notebook is an experimental laboratory. Reusable schemas and metric functions should move into `rice_dsm`; repeatable training becomes a tested script or workflow; approved artifacts are immutable and versioned.

## Debugging checklist

1. Write the shape and unit of $X$, $y$, $\boldsymbol\beta$, and $\widehat y$.
2. Recompute one fitted value and residual by hand.
3. Check row identities and split disjointness.
4. Compare against the constant baseline under the same split.
5. Inspect rank, singular values, and condition number.
6. Confirm every fitted transformer learned from training rows only.
7. Plot errors by prediction, group, time, and scientifically meaningful range.
8. Verify that every feature exists at prediction time.
9. Restart the kernel and run all cells in order.

## Professional practice

| Mathematical question | Statistical question | Software/system question |
| --- | --- | --- |
| What objective is minimized? | What population supports generalization? | Is the objective implementation tested? |
| Is the design full rank? | Does the split match future use? | Are row and schema identities recorded? |
| What do residuals reveal? | Which uncertainty matters? | Are diagnostics reproducible and monitored? |
| How does scaling transform coefficients? | Are subgroup errors acceptable? | Does one pipeline own fitted preprocessing? |

Trustworthy prediction requires all three columns. A correct equation cannot rescue invalid evidence, and good evidence cannot rescue a broken implementation.

## Application gallery: one algebra, different scientific contracts

| Application | Candidate response and features | What can go wrong |
| --- | --- | --- |
| Instrument calibration | reference quantity from sensor response | measurement error in the feature violates ordinary fixed-design assumptions |
| Spectroscopy | concentration from hundreds of wavelengths | strong collinearity, baseline drift, and instrument transfer |
| Energy forecasting | load from weather, calendar, and lags | time leakage and extrapolation during extreme weather |
| Materials degradation | later capacity from early-cycle summaries | cell, batch, chemistry, and protocol shift |
| Economics or policy | outcome from treatment and context variables | predictive association is presented as an intervention effect |

Linear regression may be a final model, a transparent baseline, or a local approximation. Its simplicity makes unsupported claims easier to see, not automatically impossible.

## Guided practice: derive and verify a different loss summary

Compute median absolute error on the existing validation predictions. Then manually compute the OLS slope on `x_board` and `y_board` after adding 100 score points to the last target.

Predict before running: which changes more, MAE or median absolute error? Why is squared-loss OLS sensitive to the altered point?

**Success criteria:** do not refit on validation data, report errors in score points, and connect the numerical change to the mathematical loss.

In [ ]:
validation_absolute_error = np.abs(validation_data[target_name].to_numpy() - validation_prediction)
validation_median_absolute_error = float(np.median(validation_absolute_error))

y_board_perturbed = y_board.copy()
y_board_perturbed[-1] += 100
perturbed_slope = (
    (x_board - x_board.mean()) @ (y_board_perturbed - y_board_perturbed.mean())
    / ((x_board - x_board.mean()) @ (x_board - x_board.mean()))
)
assert validation_median_absolute_error >= 0
assert not np.isclose(perturbed_slope, slope_by_formula)
print(f"validation median absolute error: {validation_median_absolute_error:.2f} score points")
print(f"original slope={slope_by_formula:.3f}; perturbed slope={perturbed_slope:.3f}")

## Independent practice

Choose one documented baseline feature to remove. State its meaning, measurement limitations, and availability time. Rebuild the pipeline using training data, compare on validation data with the same metrics, and inspect at least one scientifically relevant slice.

**Success criteria:** preserve the test boundary, compare with the same baseline, and separate predictive evidence from a causal interpretation.

## Extension: redesign the split

The bundled diabetes dataset lacks site, enrollment time, and repeated-patient identifiers, so it cannot support a credible site- or time-held-out experiment. Find a documented public dataset with a meaningful group or timestamp, record its license and provenance, and repeat the pipeline with group- or time-aware splitting.

**Success criteria:** justify the new unit of generalization; keep that unit disjoint; fit preprocessing inside each training boundary; and state which future population the evidence can and cannot represent.

## Retrieval practice

1. What is the difference between a residual and a future prediction error?
2. Derive $\widehat b=\bar y-\widehat w\bar x$ from the intercept normal equation.
3. Why is an explicit matrix inverse unnecessary and often undesirable?
4. Why is training error not independent evidence of generalization?
5. Why can scaling leave OLS predictions unchanged but transform coefficients?
6. What does a negative held-out $R^2$ mean?
7. Why does a small residual not make a coefficient causal?

## Takeaway

OLS selects the affine function that minimizes squared residuals on training data. The geometry and normal equations explain the fitted parameters; held-out evaluation supports a narrower claim about unseen observations. A trustworthy regression system joins that mathematics to a defensible split, a baseline, leakage-safe preprocessing, interpretable metrics, residual diagnostics, and a versioned release contract.

**Next:** we will solve the same least-squares problem iteratively. A derivative gives a downhill direction for one parameter, a gradient does so for many parameters, and a single neuron with identity activation becomes exactly this linear model.

## Further reading

- [scikit-learn: ordinary least squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- [scikit-learn: diabetes dataset description and provenance](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset)
- [scikit-learn: common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html)
- [scikit-learn: pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- [NumPy: `linalg.lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)
- [NumPy linear algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [Python tutorial: defining functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions)
- [UCI Machine Learning Repository](https://archive.ics.uci.edu/)
- [OpenML](https://www.openml.org/)
- [Data.gov](https://data.gov/), [NASA Science Data](https://science.data.nasa.gov/), and [NOAA Data Discovery](https://data.noaa.gov/onestop/)
- [World Bank Open Data](https://data.worldbank.org/) and [Google Dataset Search](https://datasetsearch.research.google.com/)

For any external dataset, record the landing page, version/date accessed, creator, license, collection process, row meaning, target construction, missingness, sensitive fields, and known limitations **before** fitting a model.